### Defining the company universe
Selecting ~35 U.S. retailers across big-box, e-commerce, and specialty retail, 
covering 2019–2023 filings. This range gives real variety in risk language and 
spans a period with genuine business disruption (COVID, inflation, supply chain) 
which matters, since we planned to use real year-over-year change to study, not flat text.

In [1]:
import pandas as pd

# Ticker symbols for the company universe — EDGAR filings are looked up
# by CIK (Central Index Key), but most tools accept ticker symbols directly
companies = {
    # Big-box / general merchandise
    "WMT": "Walmart",
    "TGT": "Target",
    "COST": "Costco",
    "BJ": "BJ's Wholesale",

    # Home improvement
    "HD": "Home Depot",
    "LOW": "Lowe's",

    # Department stores
    "M": "Macy's",
    "JWN": "Nordstrom",
    "KSS": "Kohl's",
    "DDS": "Dillard's",

    # Off-price
    "TJX": "TJX Companies",
    "ROST": "Ross Stores",
    "BURL": "Burlington Stores",

    # Specialty apparel
    "GPS": "Gap Inc.",
    "ANF": "Abercrombie & Fitch",
    "URBN": "Urban Outfitters",
    "AEO": "American Eagle Outfitters",

    # Electronics
    "BBY": "Best Buy",
    "GME": "GameStop",

    # Sporting goods
    "DKS": "Dick's Sporting Goods",

    # Home goods
    "WSM": "Williams-Sonoma",
    "RH": "RH (Restoration Hardware)",
    "BBBY": "Bed Bath & Beyond",

    # E-commerce / pure play
    "ETSY": "Etsy",
    "W": "Wayfair",
    "CHWY": "Chewy",
    "CVNA": "Carvana",

    # Discount / dollar stores
    "DG": "Dollar General",
    "DLTR": "Dollar Tree",
    "FIVE": "Five Below",

    # Grocery
    "KR": "Kroger",
    "ACI": "Albertsons",

    # Pharmacy retail
    "WBA": "Walgreens Boots Alliance",
    "CVS": "CVS Health",

    # Auto parts retail
    "AZO": "AutoZone",
    "ORLY": "O'Reilly Automotive",
}

company_df = pd.DataFrame(list(companies.items()), columns=["ticker", "company_name"])
print(f"Total companies: {len(company_df)}")
company_df.to_csv("../data/processed/company_universe.csv", index=False)
company_df

Total companies: 36


,ticker,company_name
0,WMT,Walmart
1,TGT,Target
2,COST,Costco
3,BJ,BJ's Wholesale
4,HD,Home Depot
5,LOW,Lowe's
6,M,Macy's
7,JWN,Nordstrom
8,KSS,Kohl's
9,DDS,Dillard's


### Testing filing download with one company
Before pulling filings for all 36 companies, we confirm if the download it works well for one. Walmart is a large, stable filer,
a good first test since its filings are well-formed and unlikely to expose 
edge cases we're not ready for yet.

In [2]:
import sys
sys.path.append("../src")
from edgar_client import get_downloader

dl = get_downloader(download_path="../data/raw/sec_filings")

# Download 10-K filings for Walmart within a specific date range,
dl.get(
    "10-K",
    "WMT",
    after="2019-01-01",
    before="2023-12-31",
)

print("Download complete — check data/raw/sec_filings/ for the files")

Download complete — check data/raw/sec_filings/ for the files


### Testing risk section extraction on one filing
Before scaling the download to all 36 companies, confirming we can actually 
we try to locate and extract the "Risk Factors" section from raw filing text. SEC filings aren't structured for easy 
parsing, so we need to find a reliable pattern before scaling up.

In [3]:
# Load one filing to inspect its raw structure
filing_path = "../data/raw/sec_filings/sec-edgar-filings/WMT/10-K/0000104169-23-000020/full-submission.txt"

with open(filing_path, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

print(f"Total file length: {len(raw_text):,} characters")

# Check whether "Item 1A" (the standard SEC label for Risk Factors) appears,
# and how many times — filings often reference it in a table of contents too,
# not just at the actual section start
import re
matches = [m.start() for m in re.finditer(r"Item\s+1A", raw_text, re.IGNORECASE)]
print(f"Occurrences of 'Item 1A': {len(matches)}")
print(f"Character positions: {matches[:10]}")

Total file length: 13,431,083 characters
Occurrences of 'Item 1A': 5
Character positions: [235616, 351306, 394407, 429698, 764424]


### Inspecting each "Item 1A" occurrence
Five matches means we need to distinguish the real section header from table-of-
contents references or cross-mentions elsewhere in the document. we print a
snippet around each match to see which one is the actual start of the Risk 
Factors section.

In [4]:
for i, pos in enumerate(matches):
    snippet = raw_text[pos:pos+200].replace("\n", " ")
    print(f"--- Match {i+1} at position {pos} ---")
    print(snippet)
    print()

--- Match 1 at position 235616 ---
Item 1A</a></span></div></td><td colspan="3" style="padding:2px 1pt;text-align:left;vertical-align:bottom"><div><span style="color:#0000ff;font-family:'Times New Roman',sans-serif;font-size:9pt;font-w

--- Match 2 at position 351306 ---
Item 1A. Risk Factors</a></span><span style="color:#000000;font-family:'Times New Roman',sans-serif;font-size:10pt;font-weight:400;line-height:120%">" under the sub-caption "Legal, Tax, Regulatory, Co

--- Match 3 at position 394407 ---
Item 1A</a></span><span style="color:#000000;font-family:'Times New Roman',sans-serif;font-size:10pt;font-weight:400;line-height:120%">.  With the interconnected components of this enterprise strategy

--- Match 4 at position 429698 ---
Item 1A</a></span><span style="color:#000000;font-family:'Times New Roman',sans-serif;font-size:10pt;font-weight:400;line-height:120%">. Such attacks, if successful, in addition to potential data misu

--- Match 5 at position 764424 ---
Item 1A. Risk Fac

### Stripping HTML before searching
The raw filing is HTML, not plain text, all the <span>/<div> tags were making 
the "Item 1A" matches hard to interpret. ww use BeautifulSoup to extract clean, 
readable text first, then re-running the same search on the cleaned version.

In [5]:
from bs4 import BeautifulSoup


soup = BeautifulSoup(raw_text, "lxml")
clean_text = soup.get_text(separator=" ")

print(f"Clean text length: {len(clean_text):,} characters (down from {len(raw_text):,} raw)")

# Re-run the Item 1A search on the cleaned text
matches_clean = [m.start() for m in re.finditer(r"Item\s+1A", clean_text, re.IGNORECASE)]
print(f"Occurrences of 'Item 1A' in clean text: {len(matches_clean)}")

for i, pos in enumerate(matches_clean):
    snippet = clean_text[pos:pos+200].replace("\n", " ")
    print(f"\n--- Match {i+1} at position {pos} ---")
    print(snippet)

Clean text length: 4,642,866 characters (down from 13,431,083 raw)
Occurrences of 'Item 1A' in clean text: 6

--- Match 1 at position 35338 ---
Item 1A Risk Factors 15 Item 1B Unresolved Staff Comments 27 Item 2  Properties 28 Item 3 Legal Proceedings 31 Item 4 Mine Safety Disclosures 32 Part II Item 5 Market for Registrant's Common Equity, R

--- Match 2 at position 77072 ---
Item 1A. Risk Factors " under the sub-caption "Legal, Tax, Regulatory, Compliance, Reputational and Other Risks." Environmental, Social and Governance ("ESG") Priorities Our ESG strategy is centered o

--- Match 3 at position 91413 ---
ITEM 1A. RISK FACTORS The risks described below could, in ways we may or may not be able to accurately predict, materially and adversely affect our business, results of operations, financial position 

--- Match 4 at position 93259 ---
Item 1A .  With the interconnected components of this enterprise strategy and an increasing allocation of capital expenditures focused on these init

### Extracting the actual Risk Factors section
The real section header appears in all caps ("ITEM 1A. RISK FACTORS"), 
we can distinguish it from lowercase cross-references and the table of contents 
entry. Extracting everything between that header and the start of Item 1B.

In [6]:
start_match = re.search(r"ITEM\s+1A\.\s+RISK\s+FACTORS", clean_text)

if start_match:
    start_pos = start_match.start()
    end_match = re.search(r"ITEM\s+1B\.", clean_text[start_pos:])

    if end_match:
        end_pos = start_pos + end_match.start()
        risk_section = clean_text[start_pos:end_pos]
    else:
        risk_section = clean_text[start_pos:start_pos+50000]

    print(f"Extracted risk section length: {len(risk_section):,} characters")
    print(f"\n--- First 500 characters ---")
    print(risk_section[:500])
    print(f"\n--- Last 300 characters ---")
    print(risk_section[-300:])
else:
    print("Could not find the risk factors section header — pattern needs adjustment")
    

Extracted risk section length: 78,218 characters

--- First 500 characters ---
ITEM 1A. RISK FACTORS The risks described below could, in ways we may or may not be able to accurately predict, materially and adversely affect our business, results of operations, financial position and liquidity.  Our business operations could also be affected by additional factors that apply to all companies operating in the U.S. and globally. The following risk factors do not identify all risks that we may face. Strategic Risks Failure to successfully execute our omni-channel strategy and th

--- Last 300 characters ---
ptured within our ESG reporting, which is not incorporated by reference into and does not form any part of this Annual Report on Form 10-K. A failure or perceived failure to meet our goals could adversely affect public perception of our business, associate morale or customer or shareholder support. 


### Generalization of the Extraction
Wrapping the extraction logic into a function so it can run on any filing, 
not just this one Walmart file.

In [8]:
def extract_risk_factors(filing_path: str) -> str:
    """
    Extracts the Risk Factors section (Item 1A) from a raw SEC 10-K filing.
    Returns an empty string if the section can't be found.
    """
    with open(filing_path, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()

    soup = BeautifulSoup(raw, "lxml")
    text = soup.get_text(separator=" ")

    start_match = re.search(r"ITEM\s+1A\.\s+RISK\s+FACTORS", text)
    if not start_match:
        return ""

    start_pos = start_match.start()
    end_match = re.search(r"ITEM\s+1B\.", text[start_pos:])

    if end_match:
        end_pos = start_pos + end_match.start()
        return text[start_pos:end_pos]
    else:
        return text[start_pos:start_pos+80000]


# Test the function against all 5 Walmart filings we already downloaded
import os

wmt_dir = "../data/raw/sec_filings/sec-edgar-filings/WMT/10-K"
for accession_folder in sorted(os.listdir(wmt_dir)):
    filing_file = os.path.join(wmt_dir, accession_folder, "full-submission.txt")
    section = extract_risk_factors(filing_file)
    print(f"{accession_folder}: {len(section):,} characters extracted")

0000104169-19-000016: 51,566 characters extracted
0000104169-20-000011: 55,216 characters extracted
0000104169-21-000033: 64,689 characters extracted
0000104169-22-000012: 72,773 characters extracted
0000104169-23-000020: 78,218 characters extracted


### Downloading the remaining companies
Downloading filings for the remaining 35 companies. Adding a short delay 
between requests to stay within EDGAR's rate limits.

In [9]:
import time

company_df = pd.read_csv("../data/processed/company_universe.csv")
tickers = company_df["ticker"].tolist()

failed_tickers = []

for i, ticker in enumerate(tickers):
    if ticker == "WMT":
        continue  # already downloaded

    try:
        dl.get("10-K", ticker, after="2019-01-01", before="2023-12-31")
        print(f"[{i+1}/{len(tickers)}] {ticker}: downloaded")
    except Exception as e:
        print(f"[{i+1}/{len(tickers)}] {ticker}: FAILED — {e}")
        failed_tickers.append(ticker)

    time.sleep(1)  # be polite to EDGAR's servers

print(f"\nDone. {len(failed_tickers)} failures: {failed_tickers}")

[2/36] TGT: downloaded
[3/36] COST: downloaded
[4/36] BJ: downloaded
[5/36] HD: downloaded
[6/36] LOW: downloaded
[7/36] M: downloaded
[8/36] JWN: FAILED — Ticker 'JWN' is invalid and cannot be mapped to a CIK. Please enter a valid ticker or CIK.
[9/36] KSS: downloaded
[10/36] DDS: downloaded
[11/36] TJX: downloaded
[12/36] ROST: downloaded
[13/36] BURL: downloaded
[14/36] GPS: FAILED — Ticker 'GPS' is invalid and cannot be mapped to a CIK. Please enter a valid ticker or CIK.
[15/36] ANF: downloaded
[16/36] URBN: downloaded
[17/36] AEO: downloaded
[18/36] BBY: downloaded
[19/36] GME: downloaded
[20/36] DKS: downloaded
[21/36] WSM: downloaded
[22/36] RH: downloaded
[23/36] BBBY: downloaded
[24/36] ETSY: downloaded
[25/36] W: downloaded
[26/36] CHWY: downloaded
[27/36] CVNA: downloaded
[28/36] DG: downloaded
[29/36] DLTR: downloaded
[30/36] FIVE: downloaded
[31/36] KR: downloaded
[32/36] ACI: downloaded
[33/36] WBA: FAILED — Ticker 'WBA' is invalid and cannot be mapped to a CIK. Please e

### Fixing the 3 failed downloads
JWN, GPS, and WBA are valid, well-known companies, so the failure is likely a 
ticker lookup quirk in the library, not a real data issue. We look up their 
CIK numbers directly and retrying with those instead.

In [10]:

cik_overrides = {
    "JWN": "0000072333",   # Nordstrom
    "GPS": "0000039911",   # Gap Inc.
    "WBA": "0001618921",   # Walgreens Boots Alliance
}

for ticker, cik in cik_overrides.items():
    try:
        dl.get("10-K", cik, after="2019-01-01", before="2023-12-31")
        print(f"{ticker} (CIK {cik}): downloaded")
    except Exception as e:
        print(f"{ticker} (CIK {cik}): FAILED — {e}")
    time.sleep(1)

JWN (CIK 0000072333): downloaded
GPS (CIK 0000039911): downloaded
WBA (CIK 0001618921): downloaded


### Verifying WBA's filings because it encounter a middownload error
The WBA download hit a temporary server error on one specific filing (2020) 
but still reported success overall.  So we check directly whether all 5 years 
actually downloaded, or if one is missing.

In [11]:
wba_dir = "../data/raw/sec_filings/sec-edgar-filings/0001618921/10-K"
if os.path.exists(wba_dir):
    for accession_folder in sorted(os.listdir(wba_dir)):
        print(accession_folder)
else:
    print("WBA folder not found under that path — checking alternate location")
    print(os.listdir("../data/raw/sec_filings/sec-edgar-filings/"))

0001618921-19-000069
0001618921-20-000082
0001618921-21-000085
0001618921-22-000064
0001618921-23-000062


### Checking folder naming across all companies
Some companies were downloaded by ticker, others by CIK (after the retry), 
so folder names aren't consistent. So we need to list what's actually on disk before 
building the extraction loop, so nothing gets missed.

In [12]:
sec_filings_root = "../data/raw/sec_filings/sec-edgar-filings"
downloaded_folders = sorted(os.listdir(sec_filings_root))
print(f"Total folders: {len(downloaded_folders)}")
for f in downloaded_folders:
    print(f)

Total folders: 36
0000039911
0000072333
0001618921
ACI
AEO
ANF
AZO
BBBY
BBY
BJ
BURL
CHWY
COST
CVNA
CVS
DDS
DG
DKS
DLTR
ETSY
FIVE
GME
HD
KR
KSS
LOW
M
ORLY
RH
ROST
TGT
TJX
URBN
W
WMT
WSM


### Building a folder-to-ticker mapping
Creating an explicit mapping so every company can be matched to its 
actual folder name before extracting risk sections.

In [13]:
# Map the 3 CIK-named folders back to their tickers
folder_overrides = {
    "0000039911": "GPS",
    "0000072333": "JWN",
    "0001618921": "WBA",
}

# Build the final ticker -> folder_name mapping for all 36 companies
ticker_to_folder = {}
for folder in downloaded_folders:
    if folder in folder_overrides:
        ticker_to_folder[folder_overrides[folder]] = folder
    else:
        ticker_to_folder[folder] = folder

print(f"Total mapped: {len(ticker_to_folder)}")
missing = set(company_df["ticker"]) - set(ticker_to_folder.keys())
print(f"Missing from mapping: {missing}")

# Save this mapping for reuse in later notebooks
import json
with open("../data/processed/ticker_to_folder.json", "w") as f:
    json.dump(ticker_to_folder, f, indent=2)

print("Saved ticker_to_folder.json")

Total mapped: 36
Missing from mapping: set()
Saved ticker_to_folder.json


### Extracting risk sections for all companies
Now we have to loop through every company's filings and pull out the Risk 
Factors section from each one. We're recording the company, filing year, and 
how long the extracted text is. Some filings might fail if their formatting 
is different from what we tested on Walmart, so we have to track those 
separately instead of letting one bad filing break the whole run.

In [ ]:
import sys
sys.path.append("../src")
from extract_risk_section import extract_risk_factors

results = []
extraction_failures = []

for ticker, folder_name in ticker_to_folder.items():
    company_folder = os.path.join(sec_filings_root, folder_name, "10-K")

    if not os.path.exists(company_folder):
        extraction_failures.append((ticker, "no 10-K folder found"))
        continue

    for accession_folder in sorted(os.listdir(company_folder)):
        filing_path = os.path.join(company_folder, accession_folder, "full-submission.txt")

        if not os.path.exists(filing_path):
            extraction_failures.append((ticker, f"{accession_folder}: no filing file"))
            continue

        year_code = accession_folder.split("-")[1]
        filing_year = 2000 + int(year_code)

        risk_text = extract_risk_factors(filing_path)

        if len(risk_text) < 1000:
            extraction_failures.append((ticker, f"{accession_folder}: only {len(risk_text)} chars extracted"))
            continue

        results.append({
            "ticker": ticker,
            "filing_year": filing_year,
            "accession_number": accession_folder,
            "risk_text": risk_text,
            "risk_text_length": len(risk_text),
        })

    print(f"{ticker}: processed")

print(f"\nTotal successful extractions: {len(results)}")
print(f"Total failures: {len(extraction_failures)}")
if extraction_failures:
    print("\nFailures:")
    for f in extraction_failures:
        print(f"  {f}")

GPS: processed
JWN: processed
WBA: processed
ACI: processed
AEO: processed
ANF: processed
AZO: processed
BBBY: processed
BBY: processed
BJ: processed
BURL: processed
CHWY: processed
COST: processed
CVNA: processed
CVS: processed
DDS: processed
DG: processed
DKS: processed
DLTR: processed
ETSY: processed
FIVE: processed
GME: processed
HD: processed
KR: processed
KSS: processed
LOW: processed
M: processed
ORLY: processed
RH: processed
ROST: processed
TGT: processed
TJX: processed
URBN: processed
W: processed
WMT: processed
WSM: processed

Total successful extractions: 52
Total failures: 127

Failures:
  ('GPS', '0000039911-19-000023: only 0 chars extracted')
  ('GPS', '0000039911-20-000019: only 0 chars extracted')
  ('GPS', '0000039911-21-000021: only 0 chars extracted')
  ('GPS', '0000039911-22-000012: only 0 chars extracted')
  ('GPS', '0000039911-23-000015: only 0 chars extracted')
  ('JWN', '0000072333-19-000060: only 0 chars extracted')
  ('JWN', '0000072333-20-000078: only 0 chars

### Debugging why most companies failed
We have to figure out why the extraction worked for Walmart but failed for 
almost everyone else. Our best guess is that the "ITEM 1A. RISK FACTORS" 
all-caps pattern was specific to how Walmart formats it, and other companies 
write their header differently. Checking Target's actual filing text to see 
what pattern it really uses.

In [15]:
tgt_dir = "../data/raw/sec_filings/sec-edgar-filings/TGT/10-K"
first_tgt_accession = sorted(os.listdir(tgt_dir))[0]
tgt_filing_path = os.path.join(tgt_dir, first_tgt_accession, "full-submission.txt")

with open(tgt_filing_path, "r", encoding="utf-8", errors="ignore") as f:
    tgt_raw = f.read()

tgt_soup = BeautifulSoup(tgt_raw, "lxml")
tgt_text = tgt_soup.get_text(separator=" ")

# Search case-insensitively this time, since our exact all-caps
# pattern clearly isn't universal
tgt_matches = [m.start() for m in re.finditer(r"item\s*1a", tgt_text, re.IGNORECASE)]
print(f"Case-insensitive 'item 1a' matches: {len(tgt_matches)}")

for i, pos in enumerate(tgt_matches):
    snippet = tgt_text[pos:pos+150].replace("\n", " ")
    print(f"\n--- Match {i+1} at {pos} ---")
    print(snippet)

Case-insensitive 'item 1a' matches: 3

--- Match 1 at 5157 ---
Item 1A   Risk Factors 5 Item 1B   Unresolved Staff Comments 10 Item 2   Properties 11 Item 3   Legal Proceedings 12 Item 4   Mine Safety Disclosures 

--- Match 2 at 13143 ---
Item 1A.    Risk Factors                      Our business is subject to many risks. Set forth below are the material risks we face. Risks are listed 

--- Match 3 at 96513 ---
Item 1A to this Form 10-K, which should be read in conjunction with the forward-looking statements in this report. Forward-looking statements speak on


In [16]:
import importlib
import extract_risk_section
importlib.reload(extract_risk_section)
from extract_risk_section import extract_risk_factors

In [17]:
wmt_test = extract_risk_factors(
    "../data/raw/sec_filings/sec-edgar-filings/WMT/10-K/0000104169-23-000020/full-submission.txt"
)
tgt_test = extract_risk_factors(tgt_filing_path)

print(f"Walmart: {len(wmt_test):,} characters")
print(f"Target: {len(tgt_test):,} characters")
print(f"\nTarget first 300 chars:\n{tgt_test[:300]}")

Walmart: 92,559 characters
Target: 25,534 characters

Target first 300 chars:
Item 1A.    Risk Factors                      Our business is subject to many risks. Set forth below are the material risks we face. Risks are listed in the categories where they primarily apply, but other categories may also apply. Competitive and Reputational Risks Our continued success is depende
